# EvaLoop Assignment 4: code ↔ pseudocode loop on Kaggle (no API key)

**Settings (right panel):** Accelerator = **GPU T4 x2**, Internet = **On**. Then click **Save Version → Save & Run All (Commit)** so the run continues with the browser closed. It takes roughly 3–4 h.

Runs 2 open models × {standard loop, code↔pseudocode loop} on all 378 MBPP+ tasks: greedy decoding, max 10 cycles, vLLM with each model's chat template, fp16.
Generated code runs inside this Kaggle container (sandboxed, away from your machine) through EvalPlus's `untrusted_check`.
Output: `/kaggle/working/evaloop_results.zip` (Output tab).

In [ ]:
# ---- configuration ----
FORK_URL = "https://github.com/<your-username>/EvaLooop.git"   # your public fork
MODELS = ["Qwen/Qwen2.5-Coder-7B-Instruct", "ByteDance-Seed/Seed-Coder-8B-Instruct"]
LOOPS = ["standard", "code_pseudocode"]
N_TASKS = None        # None = all 378; e.g. 150 for a seeded subset if time is short

In [ ]:
import os, subprocess, glob
!nvidia-smi --query-gpu=name,memory.total --format=csv
# Code: an uploaded Kaggle dataset containing EvaLooop-a4.tar.gz, else clone the fork
tars = glob.glob('/kaggle/input/**/EvaLooop-a4.tar.gz', recursive=True)
os.chdir('/kaggle/working')
if os.path.exists('EvaLooop'):
    pass
elif tars:
    !tar xzf {tars[0]} -C /kaggle/working
else:
    !git clone -q {FORK_URL} EvaLooop
os.chdir('/kaggle/working/EvaLooop')
!git log --oneline | head -2
!ls scripts

In [ ]:
# Dependencies (vLLM brings its own torch; ~5 min)
!pip install -q vllm "evalplus==0.3.1" fire tqdm matplotlib 2>&1 | tail -3
!python -c "import vllm, evalplus; print('vllm', vllm.__version__)"
# Download MBPP+ and compute the EvalPlus ground truth once
!python -c "import sys; sys.path.insert(0,'.'); from evaloop.evaluation.py_testing import load_mbpp_dataset; p,_=load_mbpp_dataset(); print(len(p),'tasks')"
!python -m pytest -q tests 2>&1 | tail -2

In [ ]:
import time
subset = f"--n_tasks {N_TASKS} --sample_seed 0" if N_TASKS else ""
os.makedirs('logs', exist_ok=True)
for model in MODELS:
    for loop in LOOPS:
        alias = model.split('/')[-1].lower()
        if glob.glob(f"results/{alias}__{loop}_{loop}_results.json"):
            print('already done:', alias, loop); continue
        t0 = time.time()
        cmd = (f"python scripts/run_loop.py --model {model} --loop {loop} --backend vllm_chat "
               f"--dtype half --tensor_parallel_size 2 --max_model_len 8192 "
               f"--gpu_memory_utilization 0.90 {subset} --output_dir results/ --log_level WARNING")
        print('>>>', cmd, flush=True)
        with open(f"logs/{alias}__{loop}.log", "w") as log:
            rc = subprocess.run(cmd, shell=True, stdout=log, stderr=subprocess.STDOUT).returncode
        print(open(f"logs/{alias}__{loop}.log").read()[-1500:])
        print(f"exit={rc}  minutes={(time.time()-t0)/60:.1f}", flush=True)

In [ ]:
!python scripts/analyze_extension.py --results_dir results --out_dir analysis
!cd /kaggle/working/EvaLooop && zip -qr /kaggle/working/evaloop_results.zip results logs analysis
!ls -la /kaggle/working/evaloop_results.zip